# Qwen3 0.6B Fine-Tuning — Regular Causal LM Spam Detection

Fine-tunes `Qwen/Qwen3-0.6B` as a regular causal language model on the combined
TREC-2007 + CEAS-2008 spam dataset using LoRA.

Instead of attaching a classification head, the model is trained to generate a
single label: `spam` or `ham`.


In [1]:
import hashlib
import os
import re
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from aim_tracking import create_aim_callbacks, summarize_text_classification_dataset
from datasets import ClassLabel, DatasetDict, load_dataset
from dataset.combine import combine_datasets
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed
from trl import SFTConfig, SFTTrainer

KeyboardInterrupt: 

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

SEED = 67
TRAIN_SPLIT = 0.95
VALIDATION_SPLIT = 0.006
TEST_SPLIT = 0.04
HOLDOUT_SPLIT = VALIDATION_SPLIT + TEST_SPLIT
MAX_SEQ_LENGTH = 512

# Defaults are intentionally conservative for Apple Silicon / CPU runs.
# CUDA-specific A100 overrides are applied after device detection.
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.5

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MPS_MEMORY_FRACTION = 0.92

POSITIVE_LABEL_TEXT = "spam"
NEGATIVE_LABEL_TEXT = "ham"
LABEL_TEXT_TO_ID = {
    NEGATIVE_LABEL_TEXT: 0,
    POSITIVE_LABEL_TEXT: 1,
    "valid": 0,
    "not spam": 0,
    "no": 0,
    "yes": 1,
}

AIM_EXPERIMENT_NAME = "qwen3-0.6b-spam-regular-causal-lm"
AIM_SYSTEM_TRACKING_INTERVAL = 10

set_seed(SEED)


In [ ]:
def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = find_device()
CUDA = DEVICE == "cuda"
IS_MPS = DEVICE == "mps"

if IS_MPS:
    if hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    if hasattr(torch.mps, "set_per_process_memory_fraction"):
        torch.mps.set_per_process_memory_fraction(MPS_MEMORY_FRACTION)

if CUDA:
    # A100 80GB path: use larger real batches and avoid memory-saving features
    # that slow down large CUDA GPUs.
    TRAIN_BATCH_SIZE = 24
    EVAL_BATCH_SIZE = 24
    GRADIENT_ACCUMULATION_STEPS = 1
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✅ CUDA detected: applying A100-oriented bf16 training settings.")
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

print(f"Using device: {DEVICE.upper()}")


## Dataset

Download and prepare the TREC-2007 + CEAS-2008 dataset before training.
The combined parquet is cached and reused on later runs.


In [ ]:
project_root = Path.cwd().resolve()
if not (project_root / "dataset").exists():
    project_root = project_root.parent

AIM_REPO_PATH = str(project_root)
DATASET_PATH = combine_datasets(["trec_2007", "ceas_2008"], spam_ham_ratio=0.5)


def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


DATASET_SHA256 = sha256_file(DATASET_PATH)

dataset = load_dataset("parquet", data_files=DATASET_PATH, split="train")
dataset = dataset.cast_column("label", ClassLabel(names=["valid", "spam"]))
dataset_summary = summarize_text_classification_dataset(dataset)
dataset_label_counts = dataset_summary["label_counts"]
dataset_source_counts = dataset_summary["source_counts"]
dataset_text_stats = dataset_summary["text_stats"]

print(f"Loaded {len(dataset)} rows from {DATASET_PATH}")
print(f"Dataset SHA256: {DATASET_SHA256}")
print(f"Columns: {dataset.column_names}")
print(f"Spam: {dataset_label_counts['spam']}, Ham: {dataset_label_counts['ham']}")
print(f"Source counts: {dataset_source_counts}")
print(f"Average subject chars: {dataset_text_stats['avg_subject_chars']:.2f}")
print(f"Average body chars: {dataset_text_stats['avg_body_chars']:.2f}")


Combined dataset already exists: /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/dataset/combined_datasets/generated/ceas_2008__trec_2007__dedupe_high__spam_0_5__0b6c8e960e.parquet
Loaded 81206 rows from /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/dataset/combined_datasets/generated/ceas_2008__trec_2007__dedupe_high__spam_0_5__0b6c8e960e.parquet
Dataset SHA256: 2ebcd8226c5077e9e72a72cb44637d791c3597fe473f3c9768393c72563e99a9
Columns: ['subject', 'body', 'label', 'source']
Spam: 40603, Ham: 40603
Source counts: {'trec_2007': 50019, 'ceas_2008': 31187}
Average subject chars: 41.13
Average body chars: 2232.29


In [ ]:
holdout = dataset.train_test_split(
    test_size=HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)
valid_test = holdout["test"].train_test_split(
    test_size=TEST_SPLIT / HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)

dataset = DatasetDict({
    "train": holdout["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"],
})

for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    spam = labels.count(1)
    ham = labels.count(0)
    print(f"{split}: {len(dataset[split])} rows — spam: {spam}, ham: {ham}")


train: 81124 rows — spam: 40562, ham: 40562
validation: 41 rows — spam: 21, ham: 20
test: 41 rows — spam: 20, ham: 21


## Tokenizer & Prompt Format

Qwen3 is trained as a causal LM. We format spam detection as instruction tuning:
the prompt contains the email and the target completion is a single label.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"<|im_start|> id: {tokenizer.convert_tokens_to_ids('<|im_start|>')}")
print(f"<|im_end|> id: {tokenizer.convert_tokens_to_ids('<|im_end|>')}")
print(f"<think> id: {tokenizer.convert_tokens_to_ids('<think>')}")
print(f"</think> id: {tokenizer.convert_tokens_to_ids('</think>')}")

for label_text in [POSITIVE_LABEL_TEXT, NEGATIVE_LABEL_TEXT, "valid", "yes", "no"]:
    encoded = tokenizer(label_text, add_special_tokens=False).input_ids
    print(f"{label_text!r}: token_ids={encoded}, token_count={len(encoded)}")


pad_token: '<|endoftext|>' (id=151643)
eos_token: '<|im_end|>' (id=151645)
<|im_start|> id: 151644
<|im_end|> id: 151645
<think> id: 151667
</think> id: 151668
'spam': token_ids=[75545], token_count=1
'ham': token_ids=[5604], token_count=1
'valid': token_ids=[1891], token_count=1
'yes': token_ids=[9693], token_count=1
'no': token_ids=[2152], token_count=1


In [ ]:
def build_email_text(subject: str | None, body: str | None) -> str:
    subject = (subject or "").strip()
    body = (body or "").strip()
    parts = []
    if subject:
        parts.append(f"Subject: {subject}")
    if body:
        parts.append(body)
    return "\n\n".join(parts).strip()


def build_user_prompt(email_text: str) -> str:
    return (
        "You are an email spam classifier.\n"
        f"Classify the following email as {POSITIVE_LABEL_TEXT} or {NEGATIVE_LABEL_TEXT}.\n"
        f"Return exactly one lowercase word: {POSITIVE_LABEL_TEXT} or {NEGATIVE_LABEL_TEXT}.\n\n"
        "Email:\n"
        f"{email_text}"
    )


def build_training_example(sample):
    email_text = build_email_text(sample["subject"], sample["body"])
    label_text = POSITIVE_LABEL_TEXT if int(sample["label"]) == 1 else NEGATIVE_LABEL_TEXT
    prompt_messages = [{"role": "user", "content": build_user_prompt(email_text)}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return {
        "text": email_text,
        "user_prompt": build_user_prompt(email_text),
        "prompt": prompt_text,
        "completion": f"{label_text}<|im_end|>",
        "label_text": label_text,
    }


dataset = dataset.map(build_training_example, desc="Formatting prompts")
dataset = dataset.filter(lambda sample: bool(sample["text"].strip()), desc="Filtering empty texts")

preview = dataset["train"][0]
print(preview["prompt"])
print("--- completion ---")
print(preview["completion"])


LABEL_TOKEN_IDS = {
    NEGATIVE_LABEL_TEXT: tokenizer(NEGATIVE_LABEL_TEXT, add_special_tokens=False).input_ids,
    POSITIVE_LABEL_TEXT: tokenizer(POSITIVE_LABEL_TEXT, add_special_tokens=False).input_ids,
}

for label_text, token_ids in LABEL_TOKEN_IDS.items():
    if len(token_ids) != 1:
        raise ValueError(f"Expected {label_text!r} to be one token, got {token_ids}")

LABEL_TEXT_TO_TOKEN_ID = {
    label_text: token_ids[0]
    for label_text, token_ids in LABEL_TOKEN_IDS.items()
}
LABEL_ID_TO_TEXT = {
    0: NEGATIVE_LABEL_TEXT,
    1: POSITIVE_LABEL_TEXT,
}


def label_to_id(label_text: str | None) -> int:
    if label_text is None:
        return 0
    return LABEL_TEXT_TO_ID.get(label_text, 0)


def build_classification_prompt(email_text: str) -> str:
    prompt_messages = [{"role": "user", "content": build_user_prompt(email_text.strip())}]
    return tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def tokenize_prompts_for_classification(prompt_texts, model_device):
    # Keep the assistant answer position when long emails exceed MAX_SEQ_LENGTH.
    previous_truncation_side = tokenizer.truncation_side
    tokenizer.truncation_side = "left"
    try:
        return tokenizer(
            prompt_texts,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=isinstance(prompt_texts, list),
        ).to(model_device)
    finally:
        tokenizer.truncation_side = previous_truncation_side


def tokenize_prompt_for_classification(prompt_text: str, model_device):
    return tokenize_prompts_for_classification(prompt_text, model_device)


def compute_generation_metrics(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())

    accuracy = float((predictions == labels).mean())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    specificity = tn / max(tn + fp, 1)
    balanced_accuracy = (recall + specificity) / 2

    return {
        "accuracy": accuracy,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "specificity": float(specificity),
        "balanced_accuracy": float(balanced_accuracy),
        "false_positive_count": float(fp),
        "false_negative_count": float(fn),
        "true_positive_count": float(tp),
        "true_negative_count": float(tn),
    }


def score_label_tokens(next_token_logits):
    candidate_labels = [NEGATIVE_LABEL_TEXT, POSITIVE_LABEL_TEXT]
    candidate_token_ids = torch.tensor(
        [LABEL_TEXT_TO_TOKEN_ID[label] for label in candidate_labels],
        device=next_token_logits.device,
    )
    candidate_logits = next_token_logits.index_select(-1, candidate_token_ids).float()

    if candidate_logits.ndim == 1:
        label_logits = {
            label: float(logit.detach().cpu())
            for label, logit in zip(candidate_labels, candidate_logits)
        }
        if not torch.isfinite(candidate_logits).all():
            return {
                "predicted_label": None,
                "classification_failed": True,
                "failure_reason": "non_finite_label_logits",
                "label_logits": label_logits,
                "label_probabilities": None,
            }

        candidate_probabilities = torch.softmax(candidate_logits, dim=0)
        if not torch.isfinite(candidate_probabilities).all():
            return {
                "predicted_label": None,
                "classification_failed": True,
                "failure_reason": "non_finite_label_probabilities",
                "label_logits": label_logits,
                "label_probabilities": None,
            }

        predicted_index = int(torch.argmax(candidate_logits).item())
        predicted_label = candidate_labels[predicted_index]
        return {
            "predicted_label": predicted_label,
            "classification_failed": False,
            "failure_reason": None,
            "label_logits": label_logits,
            "label_probabilities": {
                label: float(prob.detach().cpu())
                for label, prob in zip(candidate_labels, candidate_probabilities)
            },
        }

    finite_rows = torch.isfinite(candidate_logits).all(dim=1)
    safe_logits = torch.where(
        torch.isfinite(candidate_logits),
        candidate_logits,
        torch.full_like(candidate_logits, -torch.inf),
    )
    all_bad_rows = ~torch.isfinite(safe_logits).any(dim=1)
    if all_bad_rows.any():
        safe_logits[all_bad_rows, 0] = 0.0
    predictions = torch.argmax(safe_logits, dim=1)
    return predictions, finite_rows


def compute_next_token_classification_metrics(model, eval_dataset, batch_size: int | None = None):
    batch_size = batch_size or EVAL_BATCH_SIZE
    model_device = next(model.parameters()).device
    was_training = model.training
    model.eval()

    predictions = []
    labels = []
    failure_count = 0

    try:
        for start in range(0, len(eval_dataset), batch_size):
            end = min(start + batch_size, len(eval_dataset))
            batch = eval_dataset[start:end]
            prompts = [build_classification_prompt(text) for text in batch["text"]]
            inputs = tokenize_prompts_for_classification(prompts, model_device)

            with torch.no_grad():
                outputs = model(**inputs)

            attention_mask = inputs["attention_mask"]
            last_token_indices = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(len(prompts), device=model_device)
            next_token_logits = outputs.logits[batch_indices, last_token_indices]
            batch_predictions, finite_rows = score_label_tokens(next_token_logits)

            predictions.extend(batch_predictions.detach().cpu().numpy().astype(int).tolist())
            labels.extend([int(label) for label in batch["label"]])
            failure_count += int((~finite_rows).sum().detach().cpu())
    finally:
        if was_training:
            model.train()

    metrics = compute_generation_metrics(predictions, labels)
    metrics["classification_failure_count"] = float(failure_count)
    metrics["classification_failure_rate"] = failure_count / max(len(labels), 1)
    return metrics


class NextTokenClassificationMetricsCallback(TrainerCallback):
    def __init__(self, eval_dataset):
        self.eval_dataset = eval_dataset

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return

        classification_metrics = compute_next_token_classification_metrics(
            model,
            self.eval_dataset,
            batch_size=args.per_device_eval_batch_size,
        )
        metrics.update({
            f"eval_{name}": value
            for name, value in classification_metrics.items()
        })


## Model & LoRA


In [ ]:
if CUDA:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
    )

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.print_trainable_parameters()


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


## Training


In [ ]:
if CUDA:
    print("✅ CUDA detected: using full bf16 weights + adamw_torch_fused.")
    target_optim = "adamw_torch_fused"
    gradient_checkpointing = False
    gradient_checkpointing_kwargs = None
    dataloader_pin_memory = True
    dataloader_num_workers = 4
    logging_steps = 25
    output_dir = "./results/qwen3_0.6b_spam_regular_a100"
else:
    print("⚠️ CUDA not available: using conservative memory settings + adamw_torch.")
    target_optim = "adamw_torch"
    gradient_checkpointing = True
    gradient_checkpointing_kwargs = {"use_reentrant": False}
    dataloader_pin_memory = False
    dataloader_num_workers = 0
    logging_steps = 1
    output_dir = "./results/qwen3_0.6b_spam_regular_mps"

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
train_batches_per_epoch = int(np.ceil(len(dataset["train"]) / TRAIN_BATCH_SIZE))
optimizer_steps_per_epoch = int(np.ceil(train_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS))
eval_steps = max(1, optimizer_steps_per_epoch // 4)
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Train batch size: {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch}")
print(f"Total optimizer steps: {optimizer_steps_per_epoch * NUM_TRAIN_EPOCHS}")
print(f"Evaluation/save cadence: every {eval_steps} optimizer steps")
print(f"Gradient checkpointing: {gradient_checkpointing}")
print(f"Dataloader workers: {dataloader_num_workers}")
print(f"Logging steps: {logging_steps}")

training_args_kwargs = {
    "report_to": ["tensorboard"],
    "run_name": AIM_EXPERIMENT_NAME,
    "output_dir": output_dir,
    "logging_dir": f"./runs/{AIM_EXPERIMENT_NAME}",
    "logging_strategy": "steps",
    "logging_steps": logging_steps,
    "logging_first_step": True,
    "save_total_limit": 3,
    "seed": SEED,
    "data_seed": SEED,

    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "lr_scheduler_type": "cosine",

    "optim": target_optim,
    "max_steps": -1,

    "bf16": CUDA,
    "fp16": False,
    "gradient_checkpointing": gradient_checkpointing,

    "eval_strategy": "steps",
    "eval_steps": eval_steps,
    "save_strategy": "steps",
    "save_steps": eval_steps,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_f1",
    "greater_is_better": True,

    "max_length": MAX_SEQ_LENGTH,
    "completion_only_loss": True,
    "packing": False,
    "eos_token": "<|im_end|>",

    "dataloader_pin_memory": dataloader_pin_memory,
    "dataloader_num_workers": dataloader_num_workers,
    "remove_unused_columns": False,
}

if gradient_checkpointing_kwargs is not None:
    training_args_kwargs["gradient_checkpointing_kwargs"] = gradient_checkpointing_kwargs

training_args = SFTConfig(**training_args_kwargs)


In [ ]:
run_config = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "effective_batch_size": effective_batch_size,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "optim": training_args.optim,
    "device": DEVICE,
    "bf16": bool(training_args.bf16),
    "fp16": bool(training_args.fp16),
    "gradient_checkpointing": bool(training_args.gradient_checkpointing),
    "gradient_checkpointing_kwargs": gradient_checkpointing_kwargs,
    "dataloader_pin_memory": dataloader_pin_memory,
    "dataloader_num_workers": dataloader_num_workers,
    "cuda_a100_optimized": CUDA,
    "eval_steps": training_args.eval_steps,
    "save_steps": training_args.save_steps,
    "logging_steps": training_args.logging_steps,
    "tensorboard_log_dir": training_args.logging_dir,
    "output_dir": training_args.output_dir,
    "metric_for_best_model": training_args.metric_for_best_model,
    "greater_is_better": training_args.greater_is_better,
    "positive_label_text": POSITIVE_LABEL_TEXT,
    "negative_label_text": NEGATIVE_LABEL_TEXT,
}

dataset_metadata = {
    "path": DATASET_PATH,
    "sha256": DATASET_SHA256,
    "rows": sum(len(dataset[split]) for split in dataset),
    "spam": dataset_label_counts["spam"],
    "ham": dataset_label_counts["ham"],
    "sources": dataset_source_counts,
    "avg_subject_chars": dataset_text_stats["avg_subject_chars"],
    "avg_body_chars": dataset_text_stats["avg_body_chars"],
    "train_rows": len(dataset["train"]),
    "validation_rows": len(dataset["validation"]),
    "test_rows": len(dataset["test"]),
}

lora_metadata = {
    "r": LORA_R,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "task_type": "CAUSAL_LM",
}

aim_callback, notebook_aim_callback = create_aim_callbacks(
    repo_path=AIM_REPO_PATH,
    experiment_name=AIM_EXPERIMENT_NAME,
    system_tracking_interval=AIM_SYSTEM_TRACKING_INTERVAL,
    run_config=run_config,
    dataset_metadata=dataset_metadata,
    lora_metadata=lora_metadata,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    callbacks=[
        NextTokenClassificationMetricsCallback(dataset["validation"]),
        aim_callback,
        notebook_aim_callback,
    ],
)


In [ ]:
trainer_stats = trainer.train()
model = trainer.model
model.config.use_cache = True
trainer_stats


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
26,0.186584,nan


TrainOutput(global_step=26, training_loss=0.23208397867766997, metrics={'train_runtime': 412.9784, 'train_samples_per_second': 0.982, 'train_steps_per_second': 0.063, 'total_flos': 456700231680000.0, 'train_loss': 0.23208397867766997})

## Save Adapter


In [ ]:
import datetime

timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
save_path = f"./results/qwen3_0.6b_spam_regular_saved_weights_{timestamp}"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
aim_callback.experiment["final_saved_model_path"] = save_path
aim_callback.experiment["saved_tokenizer_path"] = save_path

print(f"✅ Model safely saved to disk at: {save_path}")


✅ Model safely saved to disk at: ./results/qwen3_0.6b_spam_regular_saved_weights_211719042026


## Inference

Inference uses Qwen3's hard non-thinking switch (`enable_thinking=False`) so the
model answers directly with a short label instead of reasoning first.


In [ ]:
def run_mail_classification(email_text: str):
    prompt_text = build_classification_prompt(email_text)

    model.eval()
    model_device = next(model.parameters()).device
    inputs = tokenize_prompt_for_classification(prompt_text, model_device)

    with torch.no_grad():
        outputs = model(**inputs)

    scored = score_label_tokens(outputs.logits[0, -1])
    predicted_label = scored["predicted_label"]
    classification_failed = scored["classification_failed"]

    return {
        "raw_generation": predicted_label,
        "parsed_label": predicted_label,
        "predicted_label": predicted_label,
        "parse_failed": classification_failed,
        "classification_failed": classification_failed,
        "failure_reason": scored["failure_reason"],
        "label_logits": scored["label_logits"],
        "label_probabilities": scored["label_probabilities"],
        "prompt_tokens": int(inputs["input_ids"].shape[-1]),
    }


def run_mail_from_dataset_classification(dataset_split, index: int):
    return run_mail_classification(dataset_split[index]["text"])


## Evaluation


In [ ]:
test_predictions = []
test_true_labels = []
incorrect_samples = []
classification_failures = []

for i, sample in enumerate(dataset["test"]):
    result = run_mail_classification(sample["text"])
    pred_label_text = result["predicted_label"]
    pred_label = label_to_id(pred_label_text)
    actual_label = int(sample["label"])

    test_predictions.append(pred_label)
    test_true_labels.append(actual_label)

    if result["classification_failed"]:
        classification_failures.append({
            "index": i,
            "content": sample["text"],
            "actual": LABEL_ID_TO_TEXT[actual_label],
            "failure_reason": result["failure_reason"],
            "label_logits": result["label_logits"],
            "prompt_tokens": result["prompt_tokens"],
        })

    if pred_label != actual_label:
        incorrect_samples.append({
            "index": i,
            "content": sample["text"],
            "actual": LABEL_ID_TO_TEXT[actual_label],
            "output": pred_label_text,
            "classification_failed": result["classification_failed"],
            "failure_reason": result["failure_reason"],
            "label_probabilities": result["label_probabilities"],
            "prompt_tokens": result["prompt_tokens"],
        })

rounded_metrics = {
    key: round(value, 4)
    for key, value in compute_generation_metrics(test_predictions, test_true_labels).items()
}
print(rounded_metrics)
print(f"Scoring failures: {len(classification_failures)} / {len(test_true_labels)}")
print(f"Mistakes: {len(incorrect_samples)} / {len(test_true_labels)}")


In [ ]:
print(incorrect_samples)


[{'index': 22, 'content': 'Subject: Best Price,  CialisXanaViagra\\\\/aliun, A-Z  pills, ship all countries sldmz\n\n<html>\n<head>\n<meta http-equiv="Content-Type" content="text; charset=iso-8859-1">\n</head>\n<body><center><font color=7F7F7F size=1>usedto one development ticket wife welcome considered miserable. added opened handwriting slow fascinate captain action miserable.</font><br>\n<br>\n<table border=0 cellspacing=0 cellpadding=3><tr><td bgcolor=E6F3FF align=center><font size=6 color=4FA7FF face="Century Gothic">\n<b>\nCertified <font color=0000FF>OnlinePharmacy</font><br><font color=B700B7 size=5>All Countries Shipping</b></font></font><br>\n<table border=0 cellspacing=0 cellpadding=3 width=550><tr><td width=50% valign=top bgcolor=EFEFEF align=left><font face="Century Gothic" size=3 color=000000><b>\nViagraAs</b> low as $69.95<br><b>CialisAs</b> low as $99.95<br><b>ValiumAs</b> low as $85.45<br><b>CialisSoftTabsAs</b> low as $167.50<br><b>XanaxAs</b> low as $123.45<br>plus <

In [ ]:
examples = [
    '''Subject: E-mail details of the client.
Hi Greg,
I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards,
John''',
    '''Subject: Free iPhone.
Hi Greg,
You have won a free iPhone. Press the following link to receive your reward:
"http://free-iphone.com"''',
    '''Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the Cards tab in the app
to access your new card details and start making payments online.''',
    '''Subject: Obsługa języka polskiego.
Kup najnowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym użytkownikiem!
Pozdrawiam,
Wojciech''',
    '''How are you doing? Can we meet later this afternoon?
I just wanted to check if this information is correct.''',
]

for example in examples:
    result = run_mail_classification(example)
    print("=" * 80)
    print(example)
    print(f"predicted_label={result['predicted_label']!r}")
    print(f"classification_failed={result['classification_failed']}")
    print(f"failure_reason={result['failure_reason']!r}")
    print(f"label_probabilities={result['label_probabilities']}")
    print(f"prompt_tokens={result['prompt_tokens']}")
